<a href="https://colab.research.google.com/github/lyssethpitti-debug/Modelo-Matlab/blob/main/Algoritmo_explicado_de_Simplex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Bloque de Funciones: Método Simplex con Big M

Este bloque contiene la implementación del algoritmo Simplex para resolver problemas de Programación Lineal, incorporando el método Big M para manejar restricciones con valores negativos en el lado derecho (b) o de tipo 'mayor o igual'.

In [4]:
import numpy as np

def simplex(c, A, b):
    """
    Resuelve un problema de Programación Lineal (maximización) usando el método Simplex
    con la técnica Big M para manejar restricciones >= o con b negativo.

    Args:
        c (np.array): Vector de coeficientes de la función objetivo (tamaño n_vars).
                      Debe ser para maximización.
        A (np.array): Matriz de coeficientes de las restricciones (tamaño n_constraints x n_vars).
                      Todas las restricciones se asumen inicialmente como <=.
        b (np.array): Vector de términos independientes de las restricciones (tamaño n_constraints).
                      Puede contener valores negativos.

    Returns:
        tuple: (is_feasible, x_values, z_value)
            - is_feasible (bool): True si el problema tiene una solución factible, False en caso contrario.
            - x_values (np.array): Valores de las variables originales (x1, x2, ..., xn).
                                   Devuelve None si no es factible o no se encontró solución.
            - z_value (float): Valor óptimo de la función objetivo. Devuelve None si no es factible.
    """

    print("\n--- Iniciando el algoritmo Simplex ---")

    # Paso 1: Preparación de Datos y Conversión a Forma Estándar
    # El problema se asume de maximización.
    # Las restricciones inicialmente son A_i x <= b_i.
    # Si b_i < 0, se multiplica la fila por -1, convirtiendo A_i x <= b_i en -A_i x >= -b_i.
    # Las restricciones >= requerirán variables de holgura (surplus) y artificiales.

    n_vars = len(c)  # Número de variables originales (x)
    n_constraints = len(b)  # Número de restricciones

    # M es un número muy grande para penalizar las variables artificiales.
    M = 100000.0

    # Coeficientes modificados de la función objetivo (incluyendo variables adicionales)
    c_modified = list(c)

    # Matriz A_modificada y vector b_modificado para construir el tableau
    A_modified = np.copy(A).astype(float)
    b_modified = np.copy(b).astype(float)

    # Estas listas almacenarán los índices de columna globales de las variables adicionales.
    artificial_vars_global_cols = []
    initial_artificial_basics_rows = [] # Filas donde una variable artificial es básica inicialmente

    # Listas temporales para construir la parte de la matriz A correspondiente a las variables adicionales
    A_extra_cols = [] # Cada elemento será una columna de la matriz A_extra

    current_global_var_idx_offset = n_vars # El índice de columna donde empezarán las variables adicionales

    print("Analizando restricciones y preparando el tableau inicial...")

    # Iterar sobre las restricciones para añadir variables de holgura/exceso/artificiales
    # y manejar b negativos.
    for i in range(n_constraints):
        if b_modified[i] < 0:
            # Si b_i es negativo, multiplicamos la restricción por -1.
            # Esto cambia A_i x <= b_i a -A_i x >= -b_i. Ahora es una restricción de tipo '>='.
            A_modified[i, :] *= -1
            b_modified[i] *= -1
            print(f"  Restricción {i+1}: b[{i}] era negativo. Multiplicando por -1. Ahora es una restricción '>='. Añadiendo variable de exceso (surplus) y artificial.")

            # Añadir variable de exceso (surplus) (-1 en esta restricción)
            surplus_col = np.zeros(n_constraints)
            surplus_col[i] = -1.0
            A_extra_cols.append(surplus_col)
            c_modified.append(0.0) # Coeficiente 0 para surplus en Z

            # Añadir variable artificial (+1 en esta restricción)
            artificial_col = np.zeros(n_constraints)
            artificial_col[i] = 1.0
            A_extra_cols.append(artificial_col)
            c_modified.append(-M)  # Coeficiente -M para artificial en Z (penalización Big M)

            # Almacenar el índice global de la columna de la variable artificial
            artificial_vars_global_cols.append(current_global_var_idx_offset + 1)
            initial_artificial_basics_rows.append(i)

            current_global_var_idx_offset += 2 # Se añadieron 2 variables (surplus y artificial)
        else:
            # Si b_i >= 0, y es una restricción <=, simplemente añadimos una variable de holgura (slack).
            slack_col = np.zeros(n_constraints)
            slack_col[i] = 1.0
            A_extra_cols.append(slack_col)
            c_modified.append(0.0) # Coeficiente 0 para slack en Z
            current_global_var_idx_offset += 1 # Se añadió 1 variable (slack)

    # Unir A_modified con las columnas de variables adicionales y el vector b
    if A_extra_cols:
        A_extra = np.hstack([col.reshape(-1, 1) for col in A_extra_cols])
        tableau_body = np.hstack((A_modified, A_extra, b_modified.reshape(-1, 1)))
    else:
        tableau_body = np.hstack((A_modified, b_modified.reshape(-1, 1)))

    # Crear la fila Z inicial (negativo de c_modified para maximización)
    # Si la función objetivo es Max Z = cX - M*A_k, entonces la fila Z es Z - cX + M*A_k = 0.
    # Por lo tanto, los coeficientes para las variables artificiales en la fila Z serán +M.
    z_row = np.array([-val for val in c_modified] + [0.0])

    # Paso 2: Construir el Tableau Inicial Completo
    tableau = np.vstack((tableau_body, z_row)).astype(float)

    # Paso 3: Ajustar la fila Z para eliminar coeficientes M de variables artificiales básicas
    # Las variables artificiales son básicas en su respectiva fila. Para eliminar M de Z,
    # restamos M * (fila de la restricción) a la fila Z.
    print("Ajustando la fila Z para eliminar coeficientes M de variables artificiales básicas...")
    for row_idx in initial_artificial_basics_rows:
        # Encontrar la columna de la variable artificial que es básica en esta fila.
        # Sabemos que `artificial_vars_global_cols` contiene los índices de columna globales.
        art_col_idx = -1
        for col in artificial_vars_global_cols:
            if abs(tableau[row_idx, col] - 1.0) < 1e-9: # Verificar si es la variable artificial para esta fila
                art_col_idx = col
                break

        if art_col_idx != -1:
            # Si el coeficiente de la variable artificial en la fila Z es +M, restamos M * fila_pivote.
            tableau[-1, :] -= M * tableau[row_idx, :]
            print(f"  Restando {M} * Fila {row_idx+1} a la Fila Z. Columna artificial: {art_col_idx+1}")
        else:
            print(f"  Error interno: No se encontró la variable artificial para la fila {row_idx+1}.")
            return False, None, None # Indicar un error de procesamiento

    print("Tableau inicial (después del ajuste de M):")
    print(tableau)

    # Paso 4: Iteraciones Simplex
    iteration = 0
    while True:
        iteration += 1
        print(f"\n--- Iteración {iteration} ---")

        # Criterio de Optimalidad (para maximización): Todos los coeficientes en la fila Z
        # (excluyendo el RHS) deben ser no negativos. Si son todos >= 0, hemos alcanzado la solución óptima.
        z_row_coeffs = tableau[-1, :-1] # Coeficientes de la fila Z, excluyendo el RHS
        if np.all(z_row_coeffs >= -1e-9): # Tolerancia para valores cercanos a cero
            print("¡Criterio de optimalidad alcanzado!")
            break # Solución óptima encontrada

        # Seleccionar la columna pivote (variable de entrada):
        # Se elige el coeficiente más negativo en la fila Z (para maximización).
        pivot_col_idx = np.argmin(z_row_coeffs)
        print(f"Variable de entrada (columna pivote): {pivot_col_idx+1} (coeficiente Z: {z_row_coeffs[pivot_col_idx]:.4f})")

        # Criterio de Ilimitación:
        # Si todos los elementos de la columna pivote son <= 0, el problema es ilimitado.
        if np.all(tableau[:-1, pivot_col_idx] <= 1e-9): # Tolerancia para valores cercanos a cero o negativos
            print("Problema ilimitado: todos los elementos de la columna pivote son no positivos.")
            return False, None, None # is_feasible, x_values, z_value

        # Seleccionar la fila pivote (variable de salida):
        # Se usa el criterio del ratio mínimo (RHS / elemento_columna_pivote).
        # Solo se consideran los elementos positivos del pivote en la columna.
        ratios = []
        for i in range(n_constraints):
            pivot_element = tableau[i, pivot_col_idx]
            if pivot_element > 1e-9: # Solo elementos positivos del pivote
                ratio = tableau[i, -1] / pivot_element
                ratios.append(ratio)
            else:
                ratios.append(np.inf) # No se considera para el ratio mínimo si el pivote es <= 0

        pivot_row_idx = np.argmin(ratios)
        min_ratio = ratios[pivot_row_idx]

        if min_ratio == np.inf: # Si todos los ratios son infinitos, no hay variable de salida
             print("Problema ilimitado: no se puede encontrar una variable de salida (todos los ratios son infinitos).")
             return False, None, None # is_feasible, x_values, z_value

        print(f"Variable de salida (fila pivote): {pivot_row_idx+1} (ratio mínimo: {min_ratio:.4f})")

        # Paso 5: Realizar la operación de pivoteo
        pivot_element = tableau[pivot_row_idx, pivot_col_idx]
        print(f"Elemento pivote: tableau[{pivot_row_idx+1}, {pivot_col_idx+1}] = {pivot_element:.4f}")

        # Dividir la fila pivote por el elemento pivote para hacer el pivote 1.
        tableau[pivot_row_idx, :] /= pivot_element

        # Hacer ceros todos los demás elementos de la columna pivote mediante operaciones de fila.
        for i in range(n_constraints + 1): # Incluye la fila Z
            if i != pivot_row_idx:
                factor = tableau[i, pivot_col_idx]
                tableau[i, :] -= factor * tableau[pivot_row_idx, :]

        print("Tableau después del pivoteo:")
        print(tableau)

    # Paso 6: Extracción de la Solución
    print("\n--- Extrayendo la solución ---")
    x_values = np.zeros(n_vars)

    # El número total de columnas en el tableau (excluyendo el RHS)
    total_tableau_cols_no_rhs = tableau.shape[1] - 1

    # Identificar variables básicas y sus valores
    basic_vars_indices = []
    for j in range(total_tableau_cols_no_rhs):
        col = tableau[:-1, j]
        # Una columna es una variable básica si tiene un solo 1 y el resto 0s
        if (np.sum(col == 0) == n_constraints - 1) and (np.sum(col == 1) == 1):
            row_of_one = np.where(col == 1)[0][0]
            basic_vars_indices.append((j, row_of_one)) # (columna, fila)

    for var_idx, row_idx in basic_vars_indices:
        if var_idx < n_vars: # Es una variable original x (no una slack, surplus o artificial)
            x_values[var_idx] = tableau[row_idx, -1]
            print(f"  Variable básica x_{var_idx+1} = {x_values[var_idx]:.4f}")

    # Verificar factibilidad (si alguna variable artificial es básica y positiva)
    is_feasible = True
    for art_var_col_idx in artificial_vars_global_cols:
        # Buscar si esta columna artificial es básica
        is_art_basic = False
        for var_idx, row_idx in basic_vars_indices:
            if var_idx == art_var_col_idx:
                is_art_basic = True
                # Si la variable artificial es básica y su valor es > 0, el problema es infactible
                if tableau[row_idx, -1] > 1e-9: # Tolerancia para valores cercanos a cero
                    is_feasible = False
                    print(f"  ¡Infactible! La variable artificial en columna {art_var_col_idx+1} es básica y positiva: {tableau[row_idx, -1]:.4f}")
                    break
        if not is_feasible: # Si ya encontramos que es infactible, salimos del bucle exterior
            break

    if not is_feasible:
        print("El problema de Programación Lineal no tiene una solución factible (debido a la presencia de variables artificiales en la base con valores positivos).")
        return False, None, None

    # El valor óptimo de la función objetivo Z se encuentra en la última celda del tableau
    # Para problemas de maximización, el valor de Z es el valor en tableau[-1, -1] después de las iteraciones.
    z_value = tableau[-1, -1]
    print(f"  Valor óptimo de Z = {z_value:.4f}")

    return True, x_values, z_value

### Bloque Principal (Main): Ejecución y Prueba

Aquí puedes definir los vectores `c` y `b`, y la matriz `A` para tu problema de Programación Lineal. Luego, se llamará a la función `simplex` y se mostrarán los resultados.

In [5]:
# --- Ejemplo 1: Problema Factible con b negativo ---
print("\n--- Ejecutando Ejemplo 1: Factible con b negativo ---")
# Maximizar Z = 3x1 + 5x2
# Sujeto a:
#   x1        <= 4
#   2x2       <= 12
#   3x1 + 2x2 <= 18
#   -x1 - x2  <= -6  (Originalmente x1 + x2 >= 6, pero lo convertimos a <= y luego el simplex lo maneja)
#   x1, x2 >= 0

c_example1 = np.array([3.0, 5.0])
A_example1 = np.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [3.0, 2.0],
    [-1.0, -1.0] # Para simular una restricción >=, el usuario la da como <= con b negativo
])
b_example1 = np.array([4.0, 12.0, 18.0, -6.0]) # b negativo para la última restricción

is_feasible1, x_solution1, z_optimal1 = simplex(c_example1, A_example1, b_example1)

print("\n--- Resultados Ejemplo 1 ---")
if is_feasible1:
    print(f"Problema Factible: {is_feasible1}")
    print(f"Valores de las variables (x): {x_solution1}")
    print(f"Valor óptimo de la función objetivo (Z): {z_optimal1:.4f}")
else:
    print(f"Problema Factible: {is_feasible1}")
    print("No se encontró una solución factible.")


# --- Ejemplo 2: Problema Infactible ---
print("\n\n--- Ejecutando Ejemplo 2: Infactible ---")
# Maximizar Z = x1 + x2
# Sujeto a:
#   x1 + x2 <= 2
#   x1 + x2 >= 5 (convertido a -x1 - x2 <= -5)
#   x1, x2 >= 0

c_example2 = np.array([1.0, 1.0])
A_example2 = np.array([
    [1.0, 1.0],
    [-1.0, -1.0]
])
b_example2 = np.array([2.0, -5.0])

is_feasible2, x_solution2, z_optimal2 = simplex(c_example2, A_example2, b_example2)

print("\n--- Resultados Ejemplo 2 ---")
if is_feasible2:
    print(f"Problema Factible: {is_feasible2}")
    print(f"Valores de las variables (x): {x_solution2}")
    print(f"Valor óptimo de la función objetivo (Z): {z_optimal2:.4f}")
else:
    print(f"Problema Factible: {is_feasible2}")
    print("No se encontró una solución factible.")


--- Ejecutando Ejemplo 1: Factible con b negativo ---

--- Iniciando el algoritmo Simplex ---
Analizando restricciones y preparando el tableau inicial...
  Restricción 4: b[3] era negativo. Multiplicando por -1. Ahora es una restricción '>='. Añadiendo variable de exceso (surplus) y artificial.
Ajustando la fila Z para eliminar coeficientes M de variables artificiales básicas...
  Restando 100000.0 * Fila 4 a la Fila Z. Columna artificial: 7
Tableau inicial (después del ajuste de M):
[[ 1.00000e+00  0.00000e+00  1.00000e+00  0.00000e+00  0.00000e+00
   0.00000e+00  0.00000e+00  4.00000e+00]
 [ 0.00000e+00  2.00000e+00  0.00000e+00  1.00000e+00  0.00000e+00
   0.00000e+00  0.00000e+00  1.20000e+01]
 [ 3.00000e+00  2.00000e+00  0.00000e+00  0.00000e+00  1.00000e+00
   0.00000e+00  0.00000e+00  1.80000e+01]
 [ 1.00000e+00  1.00000e+00  0.00000e+00  0.00000e+00  0.00000e+00
  -1.00000e+00  1.00000e+00  6.00000e+00]
 [-1.00003e+05 -1.00005e+05 -0.00000e+00 -0.00000e+00 -0.00000e+00
   1.00